In [1]:
"""
THE WEIGHTED NEAREST NEIGHBOUR CLASSIFIER
We looked only at k items in the vicinity of an unknown object „UO", and had a majority vote. Using the
majority vote has shown quite efficient in our previous example, but this didn't take into account the following
reasoning: The farther a neighbor is, the more it "deviates" from the "real" result. Or in other words, we can
trust the closest neighbors more than the farther ones. Let's assume, we have 11 neighbors of an unknown item
UO. The closest five neighbors belong to a class A and all the other six, which are farther away belong to a
class B. What class should be assigned to UO? The previous approach says B, because we have a 6 to 5 vote
in favor of B. On the other hand the closest 5 are all A and this should count more.

To pursue this strategy, we can assign weights to the neighbors in the following way: The nearest neighbor of
an instance gets a weight 1/1, the second closest gets a weight of 1/2 and then going on up to 1/ k for the
farthest away neighbor.

refer to the WEIGHTED NEAREST NEIGHBOUR CLASSIFIER.jpg
"""
import numpy as np
from sklearn import datasets
from collections import Counter

iris = datasets.load_iris()

data = iris.data
labels = iris.target

np.random.seed(42)
indices = np.random.permutation(len(data))

n_training_samples = 12
learn_data = data[indices[:-n_training_samples]]
learn_labels = labels[indices[:-n_training_samples]]
test_data = data[indices[-n_training_samples:]]
test_labels = labels[indices[-n_training_samples:]]

In [2]:
def distance(instance1, instance2):
    return np.linalg.norm(np.subtract(instance1, instance2))

In [3]:
def get_neighbors(training_set, labels, test_instance, k, distance):
    distances = []
    for index in range(len(training_set)):
        dist = distance(test_instance, training_set[index])
        distances.append((training_set[index], dist, labels[index]))
    
    distances.sort(key=lambda x: x[1])
    neighbors = distances[:k]
    return neighbors

def vote_harmonic_weights(neighbors, all_results=True):
    class_counter = Counter()
    number_of_neighbors = len(neighbors)
    # print(number_of_neighbors)                    # 6
    # print(neighbors)
    
    for index in range(number_of_neighbors):
        # print('class_counter[neighbors[index][2]]: ', class_counter[neighbors[index][2]]) 
        class_counter[neighbors[index][2]] += 1 / (index + 1)      # apply the formula for the: 1 + 1/2 + 1/3+ ... + 1/k
        # print('class_counter[neighbors[index][2]]: ', class_counter[neighbors[index][2]]) 
        print('class_counter[neighbors['+ str(index)+'][2]]: ', class_counter[neighbors[index][2]]) 

    # to remove np.int64 in the output they were converted to tuples
    labels, votes = zip(*class_counter.most_common())
    # print('labels: ', labels,'votes: ', votes)                  # labels:  (1,) votes:  (2.4499999999999997,)
    print('labels: ', tuple(map(int, labels)),'votes: ', votes) 
    
    winner = class_counter.most_common(1)[0][0]
    print('winner: ', winner)                                   # winner:  1
    
    votes4winner = class_counter.most_common(1)[0][1]
    print('votes4winner: ', votes4winner)                       # votes4winner:  2.4499999999999997
    
    # print('class_counter: ', class_counter)                         # Counter({1: 2.4499999999999997})
    print('class_counter: ', tuple(map(int, class_counter)) )
    print('class_counter.values(): ', class_counter.values())       # dict_values([2.4499999999999997])
    
    if all_results:
        total = sum(class_counter.values(), 0.0)
        print('Total: ', total)                                 # Total:  2.4499999999999997
        
        for key in class_counter:
            print('key: ', key)                                 # key:  1
            class_counter[key] /= total
            # print('class_counter: ', class_counter)                 # class_counter:  Counter({1: 1.0})
            print('class_counter: ', tuple(map(int, class_counter)))
        return winner, class_counter.most_common()
    else:
        return winner, votes4winner / sum(votes)

def vote_harmonic_weights(neighbors, all_results=True):
    class_counter = Counter()

    for index in range(len(neighbors)):
        # Convert NumPy scalars to Python scalars at data boundaries
        label = int(neighbors[index][2])   # FIX 
        class_counter[label] += 1 / (index + 1)

    labels, votes = zip(*class_counter.most_common())
    print('labels:', labels)
    print('votes:', votes)

    winner, votes4winner = class_counter.most_common(1)[0]
    print('winner:', winner)
    print('votes4winner:', votes4winner)

    print('class_counter:', class_counter)

    if all_results:
        total = sum(class_counter.values())
        print('Total:', total)

        for key in class_counter:
            class_counter[key] /= total

        return winner, class_counter.most_common()
    else:
        return winner, votes4winner / sum(votes)


In [4]:
# to remove np.int64 in the output they were converted to tuples

def vote_harmonic_weights(neighbors, all_results=True):
    class_counter = Counter()
    number_of_neighbors = len(neighbors)
       
    for index in range(number_of_neighbors):
        label = int(neighbors[index][2])   #  FIX 
        class_counter[label] += 1 / (index + 1)
        print('class_counter[neighbors['+ str(index)+'][2]]: ', class_counter[neighbors[index][2]]) 

    labels, votes = zip(*class_counter.most_common())
    print('labels: ', labels,'votes: ', votes)                  # labels:  (1,) votes:  (2.4499999999999997,)
        
    winner = class_counter.most_common(1)[0][0]
    print('winner: ', winner)                                   # winner:  1
    
    votes4winner = class_counter.most_common(1)[0][1]
    print('votes4winner: ', votes4winner)                       # votes4winner:  2.4499999999999997
    
    print('class_counter: ', class_counter)                         # Counter({1: 2.4499999999999997})
    print('class_counter.values(): ', class_counter.values())       # dict_values([2.4499999999999997])
    
    if all_results:
        total = sum(class_counter.values(), 0.0)
        print('Total: ', total)                                 # Total:  2.4499999999999997
        
        for key in class_counter:
            print('key: ', key)                                 # key:  1
            class_counter[key] /= total
            print('class_counter: ', class_counter)                 # class_counter:  Counter({1: 1.0})            
        return winner, class_counter.most_common()
    else:
        return winner, votes4winner / sum(votes)


In [5]:
for i in range(n_training_samples):
    neighbors = get_neighbors(learn_data, learn_labels, test_data[i], 6, distance=distance)
    print("Index:    ", i,
         ", results of votes:    ",
         vote_harmonic_weights(neighbors, all_results=True))

class_counter[neighbors[0][2]]:  1.0
class_counter[neighbors[1][2]]:  1.5
class_counter[neighbors[2][2]]:  1.8333333333333333
class_counter[neighbors[3][2]]:  2.083333333333333
class_counter[neighbors[4][2]]:  2.283333333333333
class_counter[neighbors[5][2]]:  2.4499999999999997
labels:  (1,) votes:  (2.4499999999999997,)
winner:  1
votes4winner:  2.4499999999999997
class_counter:  Counter({1: 2.4499999999999997})
class_counter.values():  dict_values([2.4499999999999997])
Total:  2.4499999999999997
key:  1
class_counter:  Counter({1: 1.0})
Index:     0 , results of votes:     (1, [(1, 1.0)])
class_counter[neighbors[0][2]]:  1.0
class_counter[neighbors[1][2]]:  1.5
class_counter[neighbors[2][2]]:  1.8333333333333333
class_counter[neighbors[3][2]]:  2.083333333333333
class_counter[neighbors[4][2]]:  2.283333333333333
class_counter[neighbors[5][2]]:  2.4499999999999997
labels:  (2,) votes:  (2.4499999999999997,)
winner:  2
votes4winner:  2.4499999999999997
class_counter:  Counter({2: 2.44